In [ ]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
from diffusers import StableDiffusionPipeline, DDIMScheduler

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
model_id = "stable-diffusion-v1-5/stable-diffusion-v1-5"

scheduler = DDIMScheduler.from_pretrained(model_id, subfolder="scheduler")
pipe = StableDiffusionPipeline.from_pretrained(
    model_id, 
    scheduler=scheduler, 
    torch_dtype=torch.float16
).to(device)

/teamspace/studios/this_studio/diffusion-brain-alignment/.venv/lib/python3.10/site-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

In [9]:
from PIL import Image
from torchvision import transforms

In [5]:
features = {}

def hook_fn(module, input, output):
    features['mid_block'] = output.detach().cpu()
    
hook_handle = pipe.unet.mid_block.register_forward_hook(hook_fn)

def preprocess_image(img):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5])  # Scale pixels to [-1, 1] for VAE
    ])
    return transform(img).unsqueeze(0).to(device, dtype=pipe.vae.dtype)


def extract_diffusion_features(image_path, timestep_val=500):
    """
    Passes an image through the VAE, adds noise corresponding to 'timestep_val',
    runs a single U-Net forward pass, and returns the bottleneck feature vector.
    """
    # Step A: Encode image into VAE latent space
    img_tensor = preprocess_image(image_path)
    with torch.no_grad():
        latents = pipe.vae.encode(img_tensor).latent_dist.sample()
        latents = latents * pipe.vae.config.scaling_factor # Standard scaling factor (0.18215)

    # Step B: Analytically add Gaussian noise for the specific timestep
    noise = torch.randn_like(latents)
    timestep = torch.tensor([timestep_val], device=device, dtype=torch.long)
    
    # Mix latent with noise based on the scheduler's alpha/beta curves
    noisy_latents = pipe.scheduler.add_noise(latents, noise, timestep)

    # Step C: Run the U-Net forward pass (unconditioned, empty text prompt)
    # We create an empty text embedding to satisfy the conditional architecture input
    with torch.no_grad():
        empty_prompt_embeds = pipe.text_encoder(
            pipe.tokenizer("", return_tensors="pt").input_ids.to(device)
        )[0]
        
        # We drop the output here because our hook catches the hidden representation mid-flight
        _ = pipe.unet(noisy_latents, timestep, encoder_hidden_states=empty_prompt_embeds)

    # Step D: Post-process the intercepted tensor
    raw_features = features['mid_block'] # Shape: [1, 1280, 8, 8]
    raw_features = raw_features.mean(dim=[2, 3])
    
    # Flatten spatial dimensions to construct a clean 1D feature vector for fMRI comparison
    # Shape transitions from [1, 1280, 8, 8] -> [1, 1280 * 8 * 8] -> [81920]
    flattened_features = torch.flatten(raw_features, start_dim=1).squeeze(0)
    
    return flattened_features

In [6]:
from diffusion_brain_alignment.data.fmri_laion import download, init_config

init_config()
download(subject="sub-01", session="ses-01")

[laion-fmri] stimuli already up to date.
[laion-fmri] stimuli already up to date.


In [2]:
from diffusion_brain_alignment.data.fmri_laion import sample_trials, load_stimuli, init_config

trials = sample_trials(subject="sub-01", session="ses-01", n_samples=10)
images = load_stimuli(trials)

ModuleNotFoundError: No module named 'diffusion_brain_alignment'

In [1]:
extract_diffusion_features(images).shape

NameError: name 'extract_diffusion_features' is not defined

In [4]:
torch.zeros((1, 1280, 16, 16)).mean(dim=[2, 3]).shape

torch.Size([1, 1280])